[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap04/cap04.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

## 💻 **Parte Prática com Exercícios de Programação**

Esta lista transforma os conceitos do Capítulo 4 em uma trilha prática de segmentação e morfologia matemática. Os EPs começam com limiarização e avançam até rotulação e descritores de componentes, sempre com matrizes pequenas para que cada pixel possa ser conferido à mão.

::: {.callout-important}
### Regra comum dos EPs morfológicos {.unnumbered}

Nas operações com vizinhança, **não faça padding**. Para cada pixel, avalie apenas as posições do elemento estruturante que caem dentro do domínio da imagem. Essa é a mesma ideia das implementações didáticas em `morph.py`, como `mm.dil0`, `mm.ero0`, `mm.dil1` e `mm.label0`: a vizinhança é recortada pelo domínio válido da imagem.
:::


---

### 🎯 Objetivo deste Caderno {.unnumbered}

O caderno permite desenvolver, validar, organizar e testar soluções de  **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.

#### Download {.unnumbered}

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:

In [2]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")

✅ Ambiente pronto. Morph: 1.1.1 | TestSuite: 1.1.0


---

#### Executando os Testes {.unnumbered}

Para rodar os testes, execute `TestSuite("EP04_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

---
### EP04_01 🎚️ Limiarização Global por Limiar Fixo
Em **scanners de documentos** e em **sistemas de leitura de código de barras**, a primeira etapa do processamento é sempre separar o que é "objeto" (tinta, texto, barras) do que é "fundo" (papel, embalagem). A **limiarização global** faz exatamente isso: compara cada pixel a um único limiar $T$ e decide, em tempo real, se ele pertence à classe clara ou à classe escura. É o operador de segmentação mais simples — e mesmo assim, está por trás de boa parte dos *pipelines* industriais de inspeção visual.
Ver na @fig-04-sim-limiar uma simulação deste EP.
#### 📋 Diretrizes de Implementação
1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Limiar:** Ler o inteiro $T$ (limiar de decisão).
3. **Dados:** Ler os valores inteiros da matriz original linha a linha.
4. **Mapeamento:** Para cada pixel $p$, calcular o novo valor pela equação:
$$p' = \begin{cases} 255, & \text{se } p \geq T \\ 0, & \text{se } p < T \end{cases}$$
5. **Saída:** Exibir a matriz binarizada com dimensões $L \times C$.
---
#### 📌 Restrições Computacionais
* **Binarização:** A saída contém **apenas** os valores $0$ ou $255$.
* **Comparação inclusiva:** O critério usa $\geq T$ (pixels iguais a $T$ tornam-se objeto).
* **Tipo:** O resultado final deve ser inteiro.
---
#### 🧠 Fundamentação Teórica
| Parâmetro | Tipo | Impacto Visual |
|-----------|------|----------------|
| **$T$ pequeno** | Inteiro | Quase tudo se torna branco (objeto cobre a imagem) |
| **$T$ grande** | Inteiro | Quase tudo se torna preto (objeto encolhe) |
| **$T$ bem escolhido** | Inteiro | Separa nitidamente objeto e fundo |
---
#### 📦 Especificação de Entrada e Saída (VPL)
**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $T$.
* Linhas seguintes: Elementos inteiros da matriz original.
**Saída:**
* Matriz binarizada em $L$ linhas e $C$ colunas, valores $0$ ou $255$ separados por espaço.
#### 📌 Exemplos
| Entrada | Saída | Observação |
|---------|-------|------------|
| 2<br>4<br>100<br>0 99 100 180<br>255 30 120 80 | 0 0 255 255<br>255 0 255 0 | $T=100$: $99<100$ vira preto, $100\geq100$ vira branco |
| 1<br>3<br>0<br>0 50 255 | 255 255 255 | $T=0$: todo pixel satisfaz $p\geq 0$ |


In [3]:
#| label: fig-04-sim-limiar
#| fig-cap: "Simulador: Limiarização Global por Limiar Fixo"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0401b" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Limiarização Global</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🎚️ p' = (p ≥ T) ? 255 : 0</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">T (Limiar)</label>
        <span id="ep0401b_vl_t" style="font-family:monospace;font-weight:bold;color:#2980b9;">128</span>
      </div>
      <input id="ep0401b_sl_t" style="width:100%;accent-color:#2980b9;" max="255" min="0" step="1" type="range" value="128">
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Entrada Original</p>
        <div id="ep0401b_grid_orig" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0401b_btn_new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Resultado (p')</p>
        <div id="ep0401b_grid_new" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
        <button id="ep0401b_btn_reset" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">↩ Resetar</button>
      </div>
    </div>
    <div id="ep0401b_debug" style="margin-top:20px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;">Fórmula: (p ≥ 128) ? 255 : 0</div>
  </div>
</div>
<script>
setTimeout(function(){
  var slT=document.getElementById('ep0401b_sl_t');
  var vlT=document.getElementById('ep0401b_vl_t');
  var gO=document.getElementById('ep0401b_grid_orig');
  var gN=document.getElementById('ep0401b_grid_new');
  var db=document.getElementById('ep0401b_debug');
  if(!slT||!gO)return;
  var pixels=[];
  function generate_0401b(){pixels=Array.from({length:16},function(){return Math.floor(Math.random()*256);});}
  function render_0401b(){
    var T=parseInt(slT.value);
    vlT.innerText=T;
    db.innerHTML='Fórmula aplicada: <b>(p ≥ '+T+') ? 255 : 0</b>';
    gO.innerHTML=''; gN.innerHTML='';
    pixels.forEach(function(p){
      var cO=document.createElement('div');
      cO.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
      cO.style.background='rgb('+p+','+p+','+p+')';
      cO.style.color=p>128?'black':'white';
      cO.innerText=p; gO.appendChild(cO);
      var res=(p>=T)?255:0;
      var cN=document.createElement('div');
      cN.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
      cN.style.background='rgb('+res+','+res+','+res+')';
      cN.style.color=res>128?'black':'white';
      cN.innerText=res; gN.appendChild(cN);
    });
  }
  slT.oninput=render_0401b;
  document.getElementById('ep0401b_btn_new').onclick=function(){generate_0401b();render_0401b();};
  document.getElementById('ep0401b_btn_reset').onclick=function(){slT.value=128;render_0401b();};
  generate_0401b(); render_0401b();
},100);
</script>
""")


In [4]:
%%writefile EP04_01.py
L = int(input())
C = int(input())
T = int(input())
img = [list(map(int, input().split())) for _ in range(L)]
for y in range(L):
    row = [255 if img[y][x] >= T else 0 for x in range(C)]
    print(*row)


Overwriting EP04_01.py


In [5]:
TestSuite("EP04_01.py").run()


---
### EP04_02 📊 Limiarização Automática de Otsu

Escolher manualmente o limiar $T$ funciona quando a iluminação é estável, mas em **microscopia digital** e em **inspeção de lâminas de sangue**, cada amostra tem um contraste diferente — um limiar fixo falharia de imagem para imagem. O **método de Otsu** resolve isso encontrando, sozinho, o limiar que **maximiza a separação estatística** entre as duas classes de pixels, tornando a segmentação automática e adaptativa.
Ver na @fig-04-sim-otsu uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões:** Ler os inteiros $L$ (linhas) e $C$ (colunas).
2. **Dados:** Ler os valores inteiros da matriz original linha a linha.
3. **Histograma:** Construir o histograma $h[i]$, $i=0,\dots,255$, contando quantos pixels têm valor $i$.
4. **Busca do limiar:** Para cada candidato $T$ de $1$ a $255$, calcular a 
**variância entre classes**:

$$
\sigma_B^2(T) = \frac{n_0 \cdot n_1}{N^2}\,(m_0 - m_1)^2
$$

onde $n_0,n_1$ são as quantidades de pixels com valor $<T$ e $\geq T$, $m_0,m_1$ são suas médias, e $N=L\times C$.

1. **Escolha:** O limiar ótimo $T^*$ é o que maximiza $\sigma_B^2(T)$ (em caso de empate, manter o **primeiro** encontrado).
2. **Aplicação:** Binarizar a imagem usando $T^*$, com o mesmo critério $p\geq T^*\Rightarrow 255$.
3. **Saída:** Exibir $T^*$ e, em seguida, a matriz binarizada.

---
#### 📌 Restrições Computacionais

* **Candidatos válidos:** Ignorar $T$ que deixe $n_0=0$ ou $n_1=0$ (classe vazia).
* **Empate:** Sempre manter o **primeiro** $T$ que atingiu o valor máximo de $\sigma_B^2$.
* **Tipo:** $T^*$ e a matriz de saída devem ser inteiros.

---
#### 🧠 Fundamentação Teórica

| Conceito | Significado | Impacto |
|----------|-------------|---------|
| **$\sigma_B^2(T)$ alta** | Classes bem separadas em $T$ | $T$ é um bom candidato a limiar |
| **Histograma bimodal** | Dois "morros" distintos | Otsu encontra o vale entre eles |
| **Histograma unimodal** | Um único "morro" | Otsu ainda escolhe *algum* $T$, mas a segmentação é pouco confiável |

---
#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linhas seguintes: Elementos inteiros da matriz original.
**Saída:**
* Linha 1: Inteiro $T^*$ (limiar ótimo encontrado).
* Linhas seguintes: Matriz binarizada em $L$ linhas e $C$ colunas, valores $0$ ou $255$.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 4<br>4<br>12 12 12 200<br>12 12 200 200<br>12 200 200 200<br>200 200 200 200 | 13<br>0 0 0 255<br>0 0 255 255<br>0 255 255 255<br>255 255 255 255 | Histograma bimodal claro: 12 e 200 |
| 1<br>2<br>10 250 | 250 | Apenas dois valores: $T^*$ fica no maior |


In [6]:
#| label: fig-04-sim-otsu
#| fig-cap: "Simulador: Limiarização Automática de Otsu"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0402" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Otsu Automático</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">📊 T* = argmax σ²ᴮ(T)</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:15px;margin-bottom:20px;">
      <svg id="ep0402_hist" height="120" style="width:100%;display:block;" viewBox="0 0 280 120"></svg>
      <p id="ep0402_info" style="text-align:center;font-size:12px;color:#555;margin-top:8px;">T* = -</p>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Entrada Original</p>
        <div id="ep0402_grid_orig" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Resultado (Otsu)</p>
        <div id="ep0402_grid_new" style="display:grid;grid-template-columns:repeat(4,42px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
    <div style="text-align:center;margin-top:15px;">
      <button id="ep0402_btn_new" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem (gera dois grupos)</button>
    </div>
  </div>
</div>
<script>
setTimeout(function(){
  var gO=document.getElementById('ep0402_grid_orig');
  var gN=document.getElementById('ep0402_grid_new');
  var info=document.getElementById('ep0402_info');
  var svg=document.getElementById('ep0402_hist');
  if(!gO)return;
  var pixels=[];
  function generate_0402(){
    var c1=20+Math.floor(Math.random()*40);
    var c2=180+Math.floor(Math.random()*60);
    pixels=[];
    for(var i=0;i<16;i++){
      var base=(Math.random()<0.5)?c1:c2;
      pixels.push(Math.max(0,Math.min(255,base+Math.floor(Math.random()*16-8))));
    }
  }
  function otsu(pix){
    var hist=new Array(256).fill(0);
    pix.forEach(function(p){hist[p]++;});
    var N=pix.length, bestVar=-1, bestT=0;
    for(var T=1;T<256;T++){
      var n0=0,s0=0;
      for(var i=0;i<T;i++){n0+=hist[i]; s0+=i*hist[i];}
      var n1=N-n0, s1=pix.reduce(function(a,b){return a+b;},0)-s0;
      if(n0===0||n1===0) continue;
      var m0=s0/n0, m1=s1/n1;
      var v=(n0*n1)*(m0-m1)*(m0-m1)/(N*N);
      if(v>bestVar){bestVar=v; bestT=T;}
    }
    return bestT;
  }
  function render_0402(){
    var T=otsu(pixels);
    info.innerHTML='T* encontrado = <b>'+T+'</b>';
    gO.innerHTML=''; gN.innerHTML='';
    var hist=new Array(256).fill(0);
    pixels.forEach(function(p){hist[p]++;});
    var maxH=Math.max.apply(null,hist);
    var bars='';
    for(var i=0;i<256;i+=4){
      var h=(hist[i]/(maxH||1))*100;
      var col=(i>=T)?'#2980b9':'#888';
      bars+='<rect x="'+(i*280/256)+'" y="'+(115-h)+'" width="4" height="'+h+'" fill="'+col+'"></rect>';
    }
    bars+='<line x1="'+(T*280/256)+'" y1="0" x2="'+(T*280/256)+'" y2="115" stroke="#e74c3c" stroke-width="2" stroke-dasharray="3,2"></line>';
    svg.innerHTML=bars;
    pixels.forEach(function(p){
      var cO=document.createElement('div');
      cO.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #eee;';
      cO.style.background='rgb('+p+','+p+','+p+')';
      cO.style.color=p>128?'black':'white';
      cO.innerText=p; gO.appendChild(cO);
      var res=(p>=T)?255:0;
      var cN=document.createElement('div');
      cN.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
      cN.style.background='rgb('+res+','+res+','+res+')';
      cN.style.color=res>128?'black':'white';
      cN.innerText=res; gN.appendChild(cN);
    });
  }
  document.getElementById('ep0402_btn_new').onclick=function(){generate_0402();render_0402();};
  generate_0402(); render_0402();
},100);
</script>
""")


In [7]:
%%writefile EP04_02.py
L = int(input())
C = int(input())
img = [list(map(int, input().split())) for _ in range(L)]
N = L * C
hist = [0] * 256
for row in img:
    for p in row:
        hist[p] += 1
melhor_var = -1
melhor_T = 0
for T in range(1, 256):
    n0 = sum(hist[:T])
    n1 = N - n0
    if n0 == 0 or n1 == 0:
        continue
    m0 = sum(i * hist[i] for i in range(T)) / n0
    m1 = sum(i * hist[i] for i in range(T, 256)) / n1
    var_entre = (n0 * n1) * (m0 - m1) ** 2 / (N ** 2)
    if var_entre > melhor_var:
        melhor_var = var_entre
        melhor_T = T
print(melhor_T)
for row in img:
    out = [255 if p >= melhor_T else 0 for p in row]
    print(*out)


Overwriting EP04_02.py


In [8]:
TestSuite("EP04_02.py").run()


---
### EP04_03 🌱 Dilatação Binária Plana (mm.dil0)

Em **microscopia de partículas** e em **OCR de placas de carro desgastadas**, traços finos ou descontínuos precisam ser "engrossados" para que o reconhecimento funcione. A **dilatação morfológica** faz exatamente isso: expande regiões claras usando um elemento estruturante $B$ — a mesma operação implementada em `morph.py` como `mm.dil0(f, B)`, usada quando $B$ é **plano** (sem pesos, só $0$/$1$).
Ver na @fig-04-sim-dil0 uma simulação deste EP.

#### 📋 Diretrizes de Implementação

1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $H_B$ (linhas) e $W_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz $f$ (a imagem original), linha a linha.
5. **Reflexão:** Construir $B_{ref}$, a versão de $B$ refletida em $180°$ (linhas e colunas invertidas) — exatamente como faz `mm.dil0` internamente.
6. **Vizinhança sem padding:** Para cada pixel $(y,x)$, percorrer as posições $(by,bx)$ de $B_{ref}$ centradas em $(y,x)$, usando o deslocamento

$$
v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{H_B}{2}+0{,}5,\quad o_x=-\tfrac{W_B}{2}+0{,}5
$$

**Descartar** todo $(v_y,v_x)$ fora de $[0,L)\times[0,C)$ — **não preencher com zeros**.

7. **Mapeamento:** Calcular cada pixel de saída como o **máximo** entre $f(y,x)$ e todos os $f(v_y,v_x)$ válidos cuja posição correspondente em $B_{ref}$ vale $1$:
$$g(y,x) = \max\Big(f(y,x),\ \max_{\substack{(v_y,v_x)\ \text{válido}\\ B_{ref}(by,bx)=1}} f(v_y,v_x)\Big)$$

8. **Saída:** Exibir a matriz $g$ com dimensões $L \times C$.

---
#### 📌 Restrições Computacionais

* **Sem padding:** Jamais inventar vizinhos fora da imagem; usar apenas os que existem de fato.
* **Reflexão obrigatória:** $B$ deve ser refletido antes de aplicado (é o que diferencia `mm.dil0` de uma simples busca por máximo).
* **Robustez de borda:** Se nenhuma posição válida de $B_{ref}=1$ cair dentro do domínio para um dado pixel, ele **mantém seu valor original**.

---

#### 🧠 Fundamentação Teórica

| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Dilatação** | $g \geq f$ sempre (extensiva) | Regiões claras crescem, buracos escuros encolhem |
| **$B$ maior** | Vizinhança mais ampla | Crescimento mais agressivo |
| **Reflexão de $B$** | $B_{ref}(y,x) = B(-y,-x)$ | Garante a definição formal de Minkowski da dilatação |

---

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $H_B$.
* Linha 4: Inteiro $W_B$.
* Próximas $H_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros da matriz $f$.

**Saída:**

* Matriz $g$ em $L$ linhas e $C$ colunas, valores inteiros separados por espaço.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>0 0 0<br>0 9 0<br>0 0 0 | 0 9 0<br>9 9 9<br>0 9 0 | $B$ em cruz simétrico: ponto isolado se expande em cruz |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 200 200 200 80 | $B$ horizontal: cada pixel "puxa" o máximo dos vizinhos da linha |


In [9]:
#| label: fig-04-sim-dil0
#| fig-cap: "Simulador: Dilatação Binária Plana (mm.dil0)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0403" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Dilatação Plana (mm.dil0)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🌱 g = f ⊕ B</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <p style="font-size:11px;font-weight:bold;color:#16a085;margin-bottom:10px;text-align:center;">Elemento Estruturante B (clique para alternar 0/1)</p>
      <div id="ep0403_grid_B" style="display:grid;grid-template-columns:repeat(3,38px);gap:4px;justify-content:center;margin-bottom:10px;"></div>
      <div style="display:flex;gap:8px;justify-content:center;">
        <button id="ep0403_btn_cross" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">➕ Cruz</button>
        <button id="ep0403_btn_box" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">⬛ Caixa</button>
        <button id="ep0403_btn_diag" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">⤫ Diagonal</button>
      </div>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Imagem Original (f)</p>
        <div id="ep0403_grid_orig" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
        <button id="ep0403_btn_new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Dilatada (g)</p>
        <div id="ep0403_grid_new" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0403_debug" style="margin-top:20px;background:#e8f5e9;border-radius:8px;padding:10px;border:1px solid #c8e6c9;font-family:monospace;font-size:11px;color:#2e7d32;text-align:center;">g(y,x) = max sobre vizinhos válidos de B refletido</div>
  </div>
</div>
<script>
setTimeout(function(){
  var gB=document.getElementById('ep0403_grid_B');
  var gO=document.getElementById('ep0403_grid_orig');
  var gN=document.getElementById('ep0403_grid_new');
  var db=document.getElementById('ep0403_debug');
  if(!gB||!gO)return;
  var B=[[0,1,0],[1,1,1],[0,1,0]];
  var L=5,C=5,pixels=[];
  function generate_0403(){
    pixels=Array.from({length:L},function(){return Array.from({length:C},function(){return (Math.random()<0.78)?0:1;});});
  }
  function reflect(M){return M.slice().reverse().map(function(r){return r.slice().reverse();});}
  function dilate(f,Bm){
    var HB=Bm.length, WB=Bm[0].length;
    var oy=-HB/2+0.5, ox=-WB/2+0.5;
    var Bref=reflect(Bm);
    var g=f.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
        if(Bref[by][bx]!==1) continue;
        var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
        if(vy>=0&&vy<L&&vx>=0&&vx<C){
          if(f[vy][vx]>g[y][x]) g[y][x]=f[vy][vx];
        }
      }
    }
    return g;
  }
  function renderB(){
    gB.innerHTML='';
    for(var by=0;by<3;by++)for(var bx=0;bx<3;bx++){
      (function(by,bx){
        var c=document.createElement('div');
        c.style.cssText='width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:13px;font-weight:bold;border-radius:4px;cursor:pointer;border:1px solid #ccc;';
        c.style.background=B[by][bx]?'#16a085':'#f5f5f5';
        c.style.color=B[by][bx]?'white':'#999';
        c.innerText=B[by][bx];
        c.onclick=function(){B[by][bx]=1-B[by][bx]; renderB(); render_0403();};
        gB.appendChild(c);
      })(by,bx);
    }
  }
  function render_0403(){
    var g=dilate(pixels,B);
    gO.innerHTML=''; gN.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var p=pixels[y][x]?255:30;
      var cO=document.createElement('div');
      cO.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #eee;';
      cO.style.background='rgb('+p+','+p+','+p+')'; cO.style.color=p>128?'black':'white';
      cO.innerText=pixels[y][x]; gO.appendChild(cO);
      var r=g[y][x]?255:30;
      var cN=document.createElement('div');
      cN.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #ddd;';
      cN.style.background='rgb('+r+','+r+','+r+')'; cN.style.color=r>128?'black':'white';
      cN.innerText=g[y][x]; gN.appendChild(cN);
    }
  }
  document.getElementById('ep0403_btn_new').onclick=function(){generate_0403();render_0403();};
  document.getElementById('ep0403_btn_cross').onclick=function(){B=[[0,1,0],[1,1,1],[0,1,0]];renderB();render_0403();};
  document.getElementById('ep0403_btn_box').onclick=function(){B=[[1,1,1],[1,1,1],[1,1,1]];renderB();render_0403();};
  document.getElementById('ep0403_btn_diag').onclick=function(){B=[[1,0,1],[0,1,0],[1,0,1]];renderB();render_0403();};
  generate_0403(); renderB(); render_0403();
},100);
</script>
""")


In [10]:
%%writefile EP04_03.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

B_ref = [row[::-1] for row in B[::-1]]
g = [row[:] for row in f]
for y in range(L):
    for x in range(C):
        for vy, vx in vizinhos(L, C, B_ref, y, x):
            if f[vy][vx] > g[y][x]:
                g[y][x] = f[vy][vx]

for row in g:
    print(*row)


Overwriting EP04_03.py


In [11]:
TestSuite("EP04_03.py").run()


---
### EP04_04 🪨 Erosão Binária Plana (mm.ero0)
Se a dilatação engrossa, a **erosão** afina. Em **sistemas de contagem de células**, ela é usada para **separar células que se tocam**: ao "comer" as bordas de cada região, conexões finas entre objetos desaparecem antes mesmo de qualquer contagem ser feita. Em `morph.py`, essa é a operação `mm.ero0(f, B)` — a **dual** exata da dilatação, e a única das duas que **não** reflete o elemento estruturante.
Ver na @fig-04-sim-ero0 uma simulação deste EP.
#### 📋 Diretrizes de Implementação
1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $H_B$ (linhas) e $W_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz $f$ (a imagem original), linha a linha.
5. **Vizinhança sem padding (sem reflexão!):** Para cada pixel $(y,x)$, percorrer as posições $(by,bx)$ de $B$ **na ordem original** (sem refletir), usando o mesmo deslocamento do EP04_03:
$$v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{H_B}{2}+0{,}5,\quad o_x=-\tfrac{W_B}{2}+0{,}5$$
**Descartar** todo $(v_y,v_x)$ fora de $[0,L)\times[0,C)$.
6. **Mapeamento:** Calcular cada pixel de saída como o **mínimo** entre $f(y,x)$ e todos os $f(v_y,v_x)$ válidos cuja posição correspondente em $B$ vale $1$:
$$g(y,x) = \min\Big(f(y,x),\ \min_{\substack{(v_y,v_x)\ \text{válido}\\ B(by,bx)=1}} f(v_y,v_x)\Big)$$
7. **Saída:** Exibir a matriz $g$ com dimensões $L \times C$.
---
#### 📌 Restrições Computacionais
* **Sem reflexão:** Diferente da dilatação, $B$ é usado **exatamente como lido** — refletir aqui seria um erro conceitual grave.
* **Sem padding:** Vizinhos fora da imagem são simplesmente ignorados, nunca tratados como $0$.
* **Robustez de borda:** Se nenhuma posição válida de $B=1$ cair dentro do domínio, o pixel mantém seu valor original.
---
#### 🧠 Fundamentação Teórica
| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Erosão** | $g \leq f$ sempre (anti-extensiva) | Regiões claras encolhem, ruído pontual desaparece |
| **Dualidade** | $\text{ero}(f,B) = -\text{dil}(-f, B_{ref})$ | Erosão e dilatação são "espelhos" matemáticos |
| **$B$ maior** | Erosão mais agressiva | Objetos finos somem completamente |
---
#### 📦 Especificação de Entrada e Saída (VPL)
**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $H_B$.
* Linha 4: Inteiro $W_B$.
* Próximas $H_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros da matriz $f$.
**Saída:**
* Matriz $g$ em $L$ linhas e $C$ colunas, valores inteiros separados por espaço.
#### 📌 Exemplos
| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>9 9 9<br>9 0 9<br>9 9 9 | 9 0 9<br>0 0 0<br>9 0 9 | O "buraco" central (0) se propaga em cruz |
| 1<br>4<br>1<br>3<br>1 1 1<br>10 200 5 80 | 10 5 5 80 | $B$ horizontal: cada pixel "puxa" o mínimo dos vizinhos da linha |


In [12]:
#| label: fig-04-sim-ero0
#| fig-cap: "Simulador: Erosão Binária Plana (mm.ero0)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0404" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Erosão Plana (mm.ero0)</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🪨 g = f ⊖ B</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <p style="font-size:11px;font-weight:bold;color:#c0392b;margin-bottom:10px;text-align:center;">Elemento Estruturante B (clique para alternar 0/1)</p>
      <div id="ep0404_grid_B" style="display:grid;grid-template-columns:repeat(3,38px);gap:4px;justify-content:center;margin-bottom:10px;"></div>
      <div style="display:flex;gap:8px;justify-content:center;">
        <button id="ep0404_btn_cross" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">➕ Cruz</button>
        <button id="ep0404_btn_box" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">⬛ Caixa</button>
        <button id="ep0404_btn_diag" style="font-size:10px;padding:4px 8px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">⤫ Diagonal</button>
      </div>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Imagem Original (f)</p>
        <div id="ep0404_grid_orig" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
        <button id="ep0404_btn_new" style="margin-top:15px;font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem</button>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:15px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:12px;">Erodida (g)</p>
        <div id="ep0404_grid_new" style="display:grid;grid-template-columns:repeat(5,36px);gap:3px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0404_debug" style="margin-top:20px;background:#fdecea;border-radius:8px;padding:10px;border:1px solid #f5c6cb;font-family:monospace;font-size:11px;color:#a93226;text-align:center;">g(y,x) = min sobre vizinhos válidos de B (sem refletir)</div>
  </div>
</div>
<script>
setTimeout(function(){
  var gB=document.getElementById('ep0404_grid_B');
  var gO=document.getElementById('ep0404_grid_orig');
  var gN=document.getElementById('ep0404_grid_new');
  if(!gB||!gO)return;
  var B=[[0,1,0],[1,1,1],[0,1,0]];
  var L=5,C=5,pixels=[];
  function generate_0404(){
    pixels=Array.from({length:L},function(){return Array.from({length:C},function(){return (Math.random()<0.78)?1:0;});});
  }
  function erode(f,Bm){
    var HB=Bm.length, WB=Bm[0].length;
    var oy=-HB/2+0.5, ox=-WB/2+0.5;
    var g=f.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
        if(Bm[by][bx]!==1) continue;
        var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
        if(vy>=0&&vy<L&&vx>=0&&vx<C){
          if(f[vy][vx]<g[y][x]) g[y][x]=f[vy][vx];
        }
      }
    }
    return g;
  }
  function renderB(){
    gB.innerHTML='';
    for(var by=0;by<3;by++)for(var bx=0;bx<3;bx++){
      (function(by,bx){
        var c=document.createElement('div');
        c.style.cssText='width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:13px;font-weight:bold;border-radius:4px;cursor:pointer;border:1px solid #ccc;';
        c.style.background=B[by][bx]?'#c0392b':'#f5f5f5';
        c.style.color=B[by][bx]?'white':'#999';
        c.innerText=B[by][bx];
        c.onclick=function(){B[by][bx]=1-B[by][bx]; renderB(); render_0404();};
        gB.appendChild(c);
      })(by,bx);
    }
  }
  function render_0404(){
    var g=erode(pixels,B);
    gO.innerHTML=''; gN.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var p=pixels[y][x]?255:30;
      var cO=document.createElement('div');
      cO.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #eee;';
      cO.style.background='rgb('+p+','+p+','+p+')'; cO.style.color=p>128?'black':'white';
      cO.innerText=pixels[y][x]; gO.appendChild(cO);
      var r=g[y][x]?255:30;
      var cN=document.createElement('div');
      cN.style.cssText='width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:bold;border-radius:3px;border:1px solid #ddd;';
      cN.style.background='rgb('+r+','+r+','+r+')'; cN.style.color=r>128?'black':'white';
      cN.innerText=g[y][x]; gN.appendChild(cN);
    }
  }
  document.getElementById('ep0404_btn_new').onclick=function(){generate_0404();render_0404();};
  document.getElementById('ep0404_btn_cross').onclick=function(){B=[[0,1,0],[1,1,1],[0,1,0]];renderB();render_0404();};
  document.getElementById('ep0404_btn_box').onclick=function(){B=[[1,1,1],[1,1,1],[1,1,1]];renderB();render_0404();};
  document.getElementById('ep0404_btn_diag').onclick=function(){B=[[1,0,1],[0,1,0],[1,0,1]];renderB();render_0404();};
  generate_0404(); renderB(); render_0404();
},100);
</script>
""")


In [13]:
%%writefile EP04_04.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

g = [row[:] for row in f]
for y in range(L):
    for x in range(C):
        for vy, vx in vizinhos(L, C, B, y, x):
            if f[vy][vx] < g[y][x]:
                g[y][x] = f[vy][vx]

for row in g:
    print(*row)


Overwriting EP04_04.py


In [14]:
TestSuite("EP04_04.py").run()


---
### EP04_05 🧹 Abertura Morfológica (Remoção de Ruído)
Imagens capturadas por **sensores de baixo custo**, como os de drones agrícolas, costumam vir salpicadas de pequenos pontos de ruído — pixels isolados que não representam nada real. Aplicar erosão seguida de dilatação com o **mesmo** elemento estruturante produz a **abertura**: ela "limpa" pontos e protuberâncias finas, mas devolve ao objeto principal praticamente seu tamanho original. É a combinação clássica usada em **pré-processamento de imagens de satélite** antes de qualquer contagem de área plantada.
Ver na @fig-04-sim-abertura uma simulação deste EP.
#### 📋 Diretrizes de Implementação
1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $H_B$ (linhas) e $W_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz binária $f$ (valores $0$ ou $1$), linha a linha.
5. **Erosão:** Calcular $e = f \ominus B$, usando exatamente o algoritmo do EP04_04 (sem refletir $B$, sem padding).
6. **Dilatação:** Calcular $g = e \oplus B$, usando exatamente o algoritmo do EP04_03 (refletindo $B$, sem padding) — mas agora aplicado sobre $e$, não sobre $f$.
7. **Saída:** Exibir a matriz resultante $g$ (a **abertura** de $f$ por $B$) com dimensões $L \times C$.
---
#### 📌 Restrições Computacionais
* **Ordem fixa:** É **sempre** erosão primeiro, depois dilatação — a ordem inversa define outro operador (fechamento, do próximo EP).
* **Mesmo $B$:** O elemento estruturante usado na erosão e na dilatação deve ser idêntico.
* **Sem padding em nenhuma das duas etapas.**
---
#### 🧠 Fundamentação Teórica
| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Anti-extensividade** | $g \subseteq f$ sempre | A abertura nunca cria pixel novo, só remove |
| **Idempotência** | $\text{abertura}(\text{abertura}(f)) = \text{abertura}(f)$ | Aplicar de novo não muda mais nada |
| **Pontos isolados** | Menores que $B$ | São completamente eliminados |
| **Núcleo do objeto** | Maior que $B$ | É recuperado quase intacto pela dilatação final |
---
#### 📦 Especificação de Entrada e Saída (VPL)
**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $H_B$.
* Linha 4: Inteiro $W_B$.
* Próximas $H_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros ($0$ ou $1$) da matriz $f$.
**Saída:**
* Matriz resultante em $L$ linhas e $C$ colunas, valores $0$ ou $1$.
#### 📌 Exemplos
| Entrada | Saída | Observação |
|---------|-------|------------|
| 7<br>7<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0<br>0 1 0 0 0 1 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 1 0<br>0 0 0 0 0 0 0<br>0 1 0 0 0 0 1 | 0 0 0 0 0 0 0<br>0 0 0 0 0 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 1 1 1 0 0<br>0 0 0 0 0 0 0<br>0 0 0 0 0 0 0 | Pontos isolados e a protuberância fina desaparecem; o quadrado central sobrevive |


In [15]:
#| label: fig-04-sim-abertura
#| fig-cap: "Simulador: Abertura Morfológica (erosão + dilatação)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0405" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Abertura Morfológica</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🧹 g = (f ⊖ B) ⊕ B</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:15px;margin-bottom:20px;text-align:center;">
      <label style="font-size:12px;font-weight:bold;color:#8e44ad;">Tamanho de B (caixa $n\\times n$)</label><br>
      <input id="ep0405_sl_n" style="width:60%;accent-color:#8e44ad;margin-top:8px;" max="5" min="3" step="2" type="range" value="3">
      <span id="ep0405_vl_n" style="font-family:monospace;font-weight:bold;color:#8e44ad;margin-left:8px;">3×3</span>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">f original</p>
        <div id="ep0405_grid_f" style="display:grid;grid-template-columns:repeat(7,28px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">e = f ⊖ B</p>
        <div id="ep0405_grid_e" style="display:grid;grid-template-columns:repeat(7,28px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">g = e ⊕ B</p>
        <div id="ep0405_grid_g" style="display:grid;grid-template-columns:repeat(7,28px);gap:2px;justify-content:center;"></div>
      </div>
    </div>
    <div style="text-align:center;margin-top:15px;">
      <button id="ep0405_btn_new" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem (com ruído)</button>
    </div>
  </div>
</div>
<script>
setTimeout(function(){
  var slN=document.getElementById('ep0405_sl_n');
  var vlN=document.getElementById('ep0405_vl_n');
  var gF=document.getElementById('ep0405_grid_f');
  var gE=document.getElementById('ep0405_grid_e');
  var gG=document.getElementById('ep0405_grid_g');
  if(!slN||!gF)return;
  var L=7,C=7,f=[];
  function generate_0405(){
    f=Array.from({length:L},function(){return new Array(C).fill(0);});
    for(var y=2;y<5;y++)for(var x=2;x<5;x++) f[y][x]=1;
    for(var k=0;k<3;k++){
      var ry=Math.floor(Math.random()*L), rx=Math.floor(Math.random()*C);
      if(f[ry][rx]===0 && (ry<1||ry>5||rx<1||rx>5)) f[ry][rx]=1;
    }
  }
  function box(n){return Array.from({length:n},function(){return new Array(n).fill(1);});}
  function erode(img,Bm){
    var HB=Bm.length, WB=Bm[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
    var g=img.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++)for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
      if(Bm[by][bx]!==1) continue;
      var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
      if(vy>=0&&vy<L&&vx>=0&&vx<C && img[vy][vx]<g[y][x]) g[y][x]=img[vy][vx];
    }
    return g;
  }
  function reflect(M){return M.slice().reverse().map(function(r){return r.slice().reverse();});}
  function dilate(img,Bm){
    var Bref=reflect(Bm);
    var HB=Bref.length, WB=Bref[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
    var g=img.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++)for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
      if(Bref[by][bx]!==1) continue;
      var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
      if(vy>=0&&vy<L&&vx>=0&&vx<C && img[vy][vx]>g[y][x]) g[y][x]=img[vy][vx];
    }
    return g;
  }
  function paint(grid,img){
    grid.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var c=document.createElement('div');
      var on=img[y][x]===1;
      c.style.cssText='width:28px;height:28px;border-radius:3px;border:1px solid #ddd;';
      c.style.background=on?'#8e44ad':'#f7f3fb';
      grid.appendChild(c);
    }
  }
  function render_0405(){
    var n=parseInt(slN.value);
    vlN.innerText=n+'×'+n;
    var B=box(n);
    var e=erode(f,B);
    var g=dilate(e,B);
    paint(gF,f); paint(gE,e); paint(gG,g);
  }
  slN.oninput=render_0405;
  document.getElementById('ep0405_btn_new').onclick=function(){generate_0405();render_0405();};
  generate_0405(); render_0405();
},100);
</script>
""")


In [16]:
%%writefile EP04_05.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

def dilata(f, B, H, W):
    B_ref = [row[::-1] for row in B[::-1]]
    g = [row[:] for row in f]
    for y in range(H):
        for x in range(W):
            for vy, vx in vizinhos(H, W, B_ref, y, x):
                if f[vy][vx] > g[y][x]:
                    g[y][x] = f[vy][vx]
    return g

def erode(f, B, H, W):
    g = [row[:] for row in f]
    for y in range(H):
        for x in range(W):
            for vy, vx in vizinhos(H, W, B, y, x):
                if f[vy][vx] < g[y][x]:
                    g[y][x] = f[vy][vx]
    return g

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

erodida = erode(f, B, L, C)
aberta = dilata(erodida, B, L, C)

for row in aberta:
    print(*row)


Overwriting EP04_05.py


In [17]:
TestSuite("EP04_05.py").run()


---
### EP04_06 🧩 Fechamento Morfológico (Preenchimento de Falhas)
Em **digitalização de impressões digitais**, sulcos da pele às vezes ficam interrompidos por sujeira ou ressecamento, criando pequenas falhas na curva contínua que deveria existir. O **fechamento** — dilatação seguida de erosão com o mesmo elemento estruturante — é o operador dual da abertura: ele **preenche buracos pequenos e reentrâncias estreitas**, sem alterar significativamente o contorno externo do objeto. É a etapa padrão antes de extrair o esqueleto de uma impressão digital.
Ver na @fig-04-sim-fechamento uma simulação deste EP.
#### 📋 Diretrizes de Implementação
1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $H_B$ (linhas) e $W_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz binária $f$ (valores $0$ ou $1$), linha a linha.
5. **Dilatação:** Calcular $d = f \oplus B$, usando exatamente o algoritmo do EP04_03 (refletindo $B$, sem padding).
6. **Erosão:** Calcular $g = d \ominus B$, usando exatamente o algoritmo do EP04_04 (sem refletir $B$, sem padding) — agora aplicado sobre $d$, não sobre $f$.
7. **Saída:** Exibir a matriz resultante $g$ (o **fechamento** de $f$ por $B$) com dimensões $L \times C$.
---
#### 📌 Restrições Computacionais
* **Ordem fixa:** É **sempre** dilatação primeiro, depois erosão — a ordem inversa é a abertura do EP04_05.
* **Mesmo $B$:** O elemento estruturante usado na dilatação e na erosão deve ser idêntico.
* **Sem padding em nenhuma das duas etapas.**
---
#### 🧠 Fundamentação Teórica
| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Extensividade** | $g \supseteq f$ sempre | O fechamento nunca remove pixel, só adiciona |
| **Idempotência** | $\text{fecha}(\text{fecha}(f)) = \text{fecha}(f)$ | Aplicar de novo não muda mais nada |
| **Buracos pequenos** | Menores que $B$ | São completamente preenchidos |
| **Dualidade** | $\text{fecha}(f) = \overline{\text{abre}(\bar f)}$ | É a abertura aplicada ao "negativo" da imagem |
---
#### 📦 Especificação de Entrada e Saída (VPL)
**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $H_B$.
* Linha 4: Inteiro $W_B$.
* Próximas $H_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros ($0$ ou $1$) da matriz $f$.
**Saída:**
* Matriz resultante em $L$ linhas e $C$ colunas, valores $0$ ou $1$.
#### 📌 Exemplos
| Entrada | Saída | Observação |
|---------|-------|------------|
| 8<br>8<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 0 1 1 0 0<br>0 0 1 1 0 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 0 0 0 0 0 0 | 0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0<br>0 0 1 1 1 1 0 0 | Os dois buracos internos não-adjacentes são totalmente preenchidos |


In [18]:
#| label: fig-04-sim-fechamento
#| fig-cap: "Simulador: Fechamento Morfológico (dilatação + erosão)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0406" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Fechamento Morfológico</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🧩 g = (f ⊕ B) ⊖ B</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:15px;margin-bottom:20px;text-align:center;">
      <label style="font-size:12px;font-weight:bold;color:#2c7a7b;">Tamanho de B (caixa $n\\times n$)</label><br>
      <input id="ep0406_sl_n" style="width:60%;accent-color:#2c7a7b;margin-top:8px;" max="5" min="3" step="2" type="range" value="3">
      <span id="ep0406_vl_n" style="font-family:monospace;font-weight:bold;color:#2c7a7b;margin-left:8px;">3×3</span>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">f original</p>
        <div id="ep0406_grid_f" style="display:grid;grid-template-columns:repeat(8,26px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">d = f ⊕ B</p>
        <div id="ep0406_grid_d" style="display:grid;grid-template-columns:repeat(8,26px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">g = d ⊖ B</p>
        <div id="ep0406_grid_g" style="display:grid;grid-template-columns:repeat(8,26px);gap:2px;justify-content:center;"></div>
      </div>
    </div>
    <div style="text-align:center;margin-top:15px;">
      <button id="ep0406_btn_new" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">🎲 Nova Imagem (com buracos)</button>
    </div>
  </div>
</div>
<script>
setTimeout(function(){
  var slN=document.getElementById('ep0406_sl_n');
  var vlN=document.getElementById('ep0406_vl_n');
  var gF=document.getElementById('ep0406_grid_f');
  var gD=document.getElementById('ep0406_grid_d');
  var gG=document.getElementById('ep0406_grid_g');
  if(!slN||!gF)return;
  var L=8,C=8,f=[];
  function generate_0406(){
    f=Array.from({length:L},function(){return new Array(C).fill(0);});
    for(var y=1;y<7;y++)for(var x=2;x<6;x++) f[y][x]=1;
    var holes=[[3,3],[4,4]];
    holes.forEach(function(h){f[h[0]][h[1]]=0;});
  }
  function box(n){return Array.from({length:n},function(){return new Array(n).fill(1);});}
  function reflect(M){return M.slice().reverse().map(function(r){return r.slice().reverse();});}
  function dilate(img,Bm){
    var Bref=reflect(Bm);
    var HB=Bref.length, WB=Bref[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
    var g=img.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++)for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
      if(Bref[by][bx]!==1) continue;
      var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
      if(vy>=0&&vy<L&&vx>=0&&vx<C && img[vy][vx]>g[y][x]) g[y][x]=img[vy][vx];
    }
    return g;
  }
  function erode(img,Bm){
    var HB=Bm.length, WB=Bm[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
    var g=img.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++)for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
      if(Bm[by][bx]!==1) continue;
      var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
      if(vy>=0&&vy<L&&vx>=0&&vx<C && img[vy][vx]<g[y][x]) g[y][x]=img[vy][vx];
    }
    return g;
  }
  function paint(grid,img){
    grid.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var c=document.createElement('div');
      var on=img[y][x]===1;
      c.style.cssText='width:26px;height:26px;border-radius:3px;border:1px solid #ddd;';
      c.style.background=on?'#2c7a7b':'#eef7f7';
      grid.appendChild(c);
    }
  }
  function render_0406(){
    var n=parseInt(slN.value);
    vlN.innerText=n+'×'+n;
    var B=box(n);
    var d=dilate(f,B);
    var g=erode(d,B);
    paint(gF,f); paint(gD,d); paint(gG,g);
  }
  slN.oninput=render_0406;
  document.getElementById('ep0406_btn_new').onclick=function(){generate_0406();render_0406();};
  generate_0406(); render_0406();
},100);
</script>
""")


In [19]:
%%writefile EP04_06.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

def dilata(f, B, H, W):
    B_ref = [row[::-1] for row in B[::-1]]
    g = [row[:] for row in f]
    for y in range(H):
        for x in range(W):
            for vy, vx in vizinhos(H, W, B_ref, y, x):
                if f[vy][vx] > g[y][x]:
                    g[y][x] = f[vy][vx]
    return g

def erode(f, B, H, W):
    g = [row[:] for row in f]
    for y in range(H):
        for x in range(W):
            for vy, vx in vizinhos(H, W, B, y, x):
                if f[vy][vx] < g[y][x]:
                    g[y][x] = f[vy][vx]
    return g

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

dilatada = dilata(f, B, L, C)
fechada = erode(dilatada, B, L, C)

for row in fechada:
    print(*row)


Overwriting EP04_06.py


In [20]:
TestSuite("EP04_06.py").run()


---
### EP04_07 ⛰️ Dilatação e Erosão com Pesos (mm.dil1 / mm.ero1)
Até agora, o elemento estruturante só dizia "este vizinho conta" ou "não conta" — mas em **modelos digitais de elevação** (usados em SIG e em planejamento de drenagem urbana), cada vizinho deveria ter um **peso diferente** dependendo da distância ou da direção do relevo. As versões **ponderadas** da dilatação e da erosão, implementadas em `morph.py` como `mm.dil1(f, b)` e `mm.ero1(f, b)`, somam (ou subtraem) o peso de cada vizinho antes de tomar o máximo (ou mínimo) — generalizando tudo o que foi feito nos EPs anteriores.
Ver na @fig-04-sim-pesos uma simulação deste EP.
#### 📋 Diretrizes de Implementação
1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $b$:** Ler os inteiros $H_b$ (linhas) e $W_b$ (colunas) do elemento estruturante ponderado.
3. **Pesos:** Ler a matriz $b$ de pesos **inteiros** (podem ser negativos, zero ou positivos), linha a linha.
4. **Dados:** Ler a matriz $f$ (a imagem original), linha a linha.
5. **Vizinhança sem padding:** Para cada pixel $(y,x)$, percorrer **todas** as posições $(by,bx)$ de $b$ (não apenas onde valeria $1$ — aqui **todo** peso participa), usando o mesmo deslocamento dos EPs anteriores:
$$v_y = y + by + o_y,\quad v_x = x + bx + o_x,\quad o_y=-\tfrac{H_b}{2}+0{,}5,\quad o_x=-\tfrac{W_b}{2}+0{,}5$$
**Descartar** todo $(v_y,v_x)$ fora de $[0,L)\times[0,C)$.
6. **Dilatação ponderada:** Calcular
$$g_{dil}(y,x) = \max\Big(f(y,x),\ \max_{(v_y,v_x)\ \text{válido}} \big(f(v_y,v_x) + b(by,bx)\big)\Big)$$
7. **Erosão ponderada:** Calcular, **usando o mesmo $b$ e sem refletir**:
$$g_{ero}(y,x) = \min\Big(f(y,x),\ \min_{(v_y,v_x)\ \text{válido}} \big(f(v_y,v_x) - b(by,bx)\big)\Big)$$
8. **Saída:** Exibir **primeiro** a matriz $g_{dil}$ completa, e **depois** a matriz $g_{ero}$ completa.
---
#### 📌 Restrições Computacionais
* **Nenhuma das duas reflete $b$** — a versão ponderada não usa reflexão, mesmo na dilatação (diferente de `mm.dil0`).
* **Todos os pesos participam:** Não existe aqui o filtro "$B=1$"; mesmo peso $0$ entra na conta.
* **Sem padding:** vizinhos fora da imagem são ignorados, nunca virtualmente preenchidos.
* **Tipo:** A saída pode conter valores negativos ou maiores que $255$ — **não** há *clipping* neste EP.
---
#### 🧠 Fundamentação Teórica
| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **Peso positivo** | "Puxa" o valor do vizinho para cima na dilatação | Simula relevo que sobe naquela direção |
| **Peso negativo** | Reduz a contribuição do vizinho | Simula distância ou atenuação direcional |
| **Dualidade ponderada** | $\text{ero1}(f,b) = -\text{dil1}(-f,b)$ | A simetria entre as duas operações se mantém mesmo com pesos |
---
#### 📦 Especificação de Entrada e Saída (VPL)
**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $H_b$.
* Linha 4: Inteiro $W_b$.
* Próximas $H_b$ linhas: elementos inteiros (podem ser negativos) da matriz $b$.
* Próximas $L$ linhas: elementos inteiros da matriz $f$.
**Saída:**
* Primeiro a matriz $g_{dil}$ em $L$ linhas e $C$ colunas.
* Em seguida a matriz $g_{ero}$ em $L$ linhas e $C$ colunas.
#### 📌 Exemplos
| Entrada | Saída | Observação |
|---------|-------|------------|
| 3<br>3<br>3<br>3<br>0 1 0<br>1 2 1<br>0 1 0<br>10 20 30<br>40 50 60<br>70 80 90 | 50 60 61<br>80 90 91<br>81 91 92<br>8 9 19<br>9 10 20<br>39 40 50 | Peso central $2$ acelera o crescimento na dilatação e o encolhimento na erosão |


In [21]:
#| label: fig-04-sim-pesos
#| fig-cap: "Simulador: Dilatação e Erosão com Pesos (mm.dil1 / mm.ero1)"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0407" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Pesos no Elemento Estruturante</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">⛰️ dil1 / ero1</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <p style="font-size:11px;font-weight:bold;color:#d35400;margin-bottom:10px;text-align:center;">Pesos b (clique e arraste o slider de cada célula)</p>
      <div id="ep0407_grid_b" style="display:grid;grid-template-columns:repeat(3,56px);gap:6px;justify-content:center;"></div>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">f original</p>
        <div id="ep0407_grid_f" style="display:grid;grid-template-columns:repeat(3,52px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#27ae60;text-transform:uppercase;margin-bottom:10px;">dil1(f,b)</p>
        <div id="ep0407_grid_d" style="display:grid;grid-template-columns:repeat(3,52px);gap:4px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#c0392b;text-transform:uppercase;margin-bottom:10px;">ero1(f,b)</p>
        <div id="ep0407_grid_e" style="display:grid;grid-template-columns:repeat(3,52px);gap:4px;justify-content:center;"></div>
      </div>
    </div>
  </div>
</div>
<script>
setTimeout(function(){
  var gB=document.getElementById('ep0407_grid_b');
  var gF=document.getElementById('ep0407_grid_f');
  var gD=document.getElementById('ep0407_grid_d');
  var gE=document.getElementById('ep0407_grid_e');
  if(!gB||!gF)return;
  var b=[[0,1,0],[1,2,1],[0,1,0]];
  var f=[[10,20,30],[40,50,60],[70,80,90]];
  var L=3,C=3;
  function compute(){
    var oy=-3/2+0.5, ox=-3/2+0.5;
    var dil=f.map(function(r){return r.slice();});
    var ero=f.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      for(var by=0;by<3;by++)for(var bx=0;bx<3;bx++){
        var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
        if(vy>=0&&vy<L&&vx>=0&&vx<C){
          var cd=f[vy][vx]+b[by][bx];
          if(cd>dil[y][x]) dil[y][x]=cd;
          var ce=f[vy][vx]-b[by][bx];
          if(ce<ero[y][x]) ero[y][x]=ce;
        }
      }
    }
    return {dil:dil,ero:ero};
  }
  function renderB(){
    gB.innerHTML='';
    for(var by=0;by<3;by++)for(var bx=0;bx<3;bx++){
      (function(by,bx){
        var wrap=document.createElement('div');
        wrap.style.cssText='display:flex;flex-direction:column;align-items:center;';
        var val=document.createElement('div');
        val.style.cssText='font-family:monospace;font-weight:bold;font-size:11px;color:#d35400;margin-bottom:2px;';
        val.innerText=b[by][bx];
        var sl=document.createElement('input');
        sl.type='range'; sl.min='-5'; sl.max='5'; sl.step='1'; sl.value=b[by][bx];
        sl.style.cssText='width:50px;accent-color:#d35400;';
        sl.oninput=function(){b[by][bx]=parseInt(sl.value); val.innerText=b[by][bx]; render_0407();};
        wrap.appendChild(val); wrap.appendChild(sl);
        gB.appendChild(wrap);
      })(by,bx);
    }
  }
  function paint(grid,img,color){
    grid.innerHTML='';
    var maxAbs=Math.max.apply(null,img.flat().map(Math.abs))||1;
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var c=document.createElement('div');
      var v=img[y][x];
      var inten=Math.min(255,Math.max(0,v));
      c.style.cssText='width:52px;height:42px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:bold;border-radius:4px;border:1px solid #ddd;';
      c.style.background='rgb('+inten+','+inten+','+inten+')';
      c.style.color=inten>128?'black':'white';
      c.innerText=v;
      grid.appendChild(c);
    }
  }
  function render_0407(){
    var res=compute();
    paint(gF,f); paint(gD,res.dil); paint(gE,res.ero);
  }
  renderB(); render_0407();
},100);
</script>
""")


In [22]:
%%writefile EP04_07.py
def vizinhos(H, W, b, y, x):
    HB, WB = len(b), len(b[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W:
                yield vy, vx, b[by][bx]

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
b = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

dil = [row[:] for row in f]
ero = [row[:] for row in f]
for y in range(L):
    for x in range(C):
        for vy, vx, p in vizinhos(L, C, b, y, x):
            cand_d = f[vy][vx] + p
            if cand_d > dil[y][x]:
                dil[y][x] = cand_d
            cand_e = f[vy][vx] - p
            if cand_e < ero[y][x]:
                ero[y][x] = cand_e

for row in dil:
    print(*row)
for row in ero:
    print(*row)


Overwriting EP04_07.py


In [23]:
TestSuite("EP04_07.py").run()


---
### EP04_08 🌋 Gradiente Morfológico, Top-hat e Black-hat
Em **inspeção automática de placas de circuito**, três perguntas aparecem o tempo todo: onde estão as **bordas** dos componentes? Quais **detalhes claros e pequenos** (como pontos de solda) se destacam do fundo? Quais **reentrâncias escuras** (como fissuras) o fundo esconde? Um único par erosão/dilatação responde às três: o **gradiente morfológico** evidencia contornos, o **top-hat** revela picos estreitos, e o **black-hat** revela vales estreitos — três ferramentas, uma só vizinhança.
Ver na @fig-04-sim-gradiente uma simulação deste EP.
#### 📋 Diretrizes de Implementação
1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $H_B$ (linhas) e $W_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz $f$ (a imagem original, em tons de cinza), linha a linha.
5. **Operadores de base:** Calcular, exatamente como nos EPs 04_03 a 04_06:
   * $d = f \oplus B$ (dilatação),
   * $e = f \ominus B$ (erosão),
   * $\text{abertura} = e \oplus B$,
   * $\text{fechamento} = d \ominus B$.
6. **Gradiente morfológico:** $\text{grad}(y,x) = d(y,x) - e(y,x)$.
7. **Top-hat:** $\text{tophat}(y,x) = f(y,x) - \text{abertura}(y,x)$.
8. **Black-hat:** $\text{blackhat}(y,x) = \text{fechamento}(y,x) - f(y,x)$.
9. **Saída:** Exibir, **nesta ordem**, as três matrizes completas: gradiente, top-hat, black-hat.
---
#### 📌 Restrições Computacionais
* **Sem padding em nenhuma etapa intermediária** — dilatação, erosão, abertura e fechamento seguem as mesmas regras de vizinhança dos EPs anteriores.
* **Não há *clipping*:** as três saídas podem conter qualquer valor inteiro (o gradiente é sempre $\geq 0$, mas top-hat e black-hat também).
* **Reaproveitamento:** $d$ e $e$ devem ser calculados **uma única vez** e reaproveitados para montar abertura, fechamento e gradiente.
---
#### 🧠 Fundamentação Teórica
| Operador | Fórmula | O que revela |
|----------|---------|----------------|
| **Gradiente** | $d - e$ | Bordas: zero em regiões planas, alto nas transições |
| **Top-hat** | $f - \text{abertura}(f)$ | Elementos **claros e finos**, menores que $B$ |
| **Black-hat** | $\text{fechamento}(f) - f$ | Elementos **escuros e finos**, menores que $B$ |
---
#### 📦 Especificação de Entrada e Saída (VPL)
**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $H_B$.
* Linha 4: Inteiro $W_B$.
* Próximas $H_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros da matriz $f$.
**Saída:**
* Matriz gradiente em $L$ linhas e $C$ colunas.
* Matriz top-hat em $L$ linhas e $C$ colunas.
* Matriz black-hat em $L$ linhas e $C$ colunas.
#### 📌 Exemplos
| Entrada | Saída | Observação |
|---------|-------|------------|
| 9<br>9<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 80 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 2 10 10<br>10 10 10 10 10 10 10 10 10<br>10 10 10 10 10 10 10 10 10 | (gradiente: halo $3\times3=70$ em torno de $(2,2)$ e halo $3\times3=8$ em torno de $(6,6)$, resto $0$)<br>(top-hat: único $70$ em $(2,2)$, resto $0$)<br>(black-hat: único $8$ em $(6,6)$, resto $0$) | Pico isolado vira top-hat; vale isolado vira black-hat; ambos aparecem no gradiente |


In [24]:
#| label: fig-04-sim-gradiente
#| fig-cap: "Simulador: Gradiente Morfológico, Top-hat e Black-hat"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0408" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Gradiente / Top-hat / Black-hat</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🌋 3 operadores, 1 vizinhança</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="text-align:center;margin-bottom:14px;">
      <button id="ep0408_btn_pico" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #ccc;background:#fdf2e9;cursor:pointer;margin-right:6px;">☀️ Adicionar Pico</button>
      <button id="ep0408_btn_vale" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #ccc;background:#eaf2fb;cursor:pointer;margin-right:6px;">🕳️ Adicionar Vale</button>
      <button id="ep0408_btn_reset" style="font-size:11px;padding:6px 12px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">↩ Limpar</button>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:10px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:8px;">f</p>
        <div id="ep0408_grid_f" style="display:grid;grid-template-columns:repeat(9,20px);gap:1px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#6c3483;text-transform:uppercase;margin-bottom:8px;">gradiente</p>
        <div id="ep0408_grid_grad" style="display:grid;grid-template-columns:repeat(9,20px);gap:1px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#d35400;text-transform:uppercase;margin-bottom:8px;">top-hat</p>
        <div id="ep0408_grid_th" style="display:grid;grid-template-columns:repeat(9,20px);gap:1px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:10px;border-radius:12px;">
        <p style="font-size:9px;font-weight:600;color:#2980b9;text-transform:uppercase;margin-bottom:8px;">black-hat</p>
        <div id="ep0408_grid_bh" style="display:grid;grid-template-columns:repeat(9,20px);gap:1px;justify-content:center;"></div>
      </div>
    </div>
  </div>
</div>
<script>
setTimeout(function(){
  var gF=document.getElementById('ep0408_grid_f');
  var gGrad=document.getElementById('ep0408_grid_grad');
  var gTh=document.getElementById('ep0408_grid_th');
  var gBh=document.getElementById('ep0408_grid_bh');
  if(!gF)return;
  var L=9,C=9,f=[],B=[[1,1,1],[1,1,1],[1,1,1]];
  function reset_0408(){ f=Array.from({length:L},function(){return new Array(C).fill(10);}); }
  function morph(img,Bm,mode){
    var Bref=mode==='dil'?Bm.slice().reverse().map(function(r){return r.slice().reverse();}):Bm;
    var HB=Bref.length, WB=Bref[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
    var g=img.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++)for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
      if(Bref[by][bx]!==1) continue;
      var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
      if(vy>=0&&vy<L&&vx>=0&&vx<C){
        if(mode==='dil' && img[vy][vx]>g[y][x]) g[y][x]=img[vy][vx];
        if(mode==='ero' && img[vy][vx]<g[y][x]) g[y][x]=img[vy][vx];
      }
    }
    return g;
  }
  function paint(grid,img,cmin,cmax,hue){
    grid.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var v=img[y][x];
      var t=cmax>cmin?(v-cmin)/(cmax-cmin):0;
      var inten=Math.round(255-t*200);
      var c=document.createElement('div');
      c.style.cssText='width:20px;height:20px;border-radius:2px;';
      c.style.background= v===0 ? '#f2f2f2' : hue;
      c.style.opacity = v===0 ? '1' : (0.35+0.65*Math.min(1,t));
      grid.appendChild(c);
    }
  }
  function render_0408(){
    var d=morph(f,B,'dil');
    var e=morph(f,B,'ero');
    var ab=morph(e,B,'dil');
    var fc=morph(d,B,'ero');
    var grad=f.map(function(r,y){return r.map(function(_,x){return d[y][x]-e[y][x];});});
    var th=f.map(function(r,y){return r.map(function(v,x){return v-ab[y][x];});});
    var bh=f.map(function(r,y){return r.map(function(v,x){return fc[y][x]-v;});});
    var maxF=Math.max.apply(null,f.flat());
    paint(gF,f,10,maxF,'#7f8c8d');
    paint(gGrad,grad,0,Math.max(1,Math.max.apply(null,grad.flat())),'#6c3483');
    paint(gTh,th,0,Math.max(1,Math.max.apply(null,th.flat())),'#d35400');
    paint(gBh,bh,0,Math.max(1,Math.max.apply(null,bh.flat())),'#2980b9');
  }
  document.getElementById('ep0408_btn_pico').onclick=function(){
    var y=2+Math.floor(Math.random()*5), x=2+Math.floor(Math.random()*5);
    f[y][x]=Math.min(255,f[y][x]+60+Math.floor(Math.random()*30));
    render_0408();
  };
  document.getElementById('ep0408_btn_vale').onclick=function(){
    var y=2+Math.floor(Math.random()*5), x=2+Math.floor(Math.random()*5);
    f[y][x]=Math.max(0,f[y][x]-8-Math.floor(Math.random()*4));
    render_0408();
  };
  document.getElementById('ep0408_btn_reset').onclick=function(){reset_0408();render_0408();};
  reset_0408();
  f[2][2]=80; f[6][6]=2;
  render_0408();
},100);
</script>
""")


In [25]:
%%writefile EP04_08.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

def dilata(f, B, H, W):
    B_ref = [row[::-1] for row in B[::-1]]
    g = [row[:] for row in f]
    for y in range(H):
        for x in range(W):
            for vy, vx in vizinhos(H, W, B_ref, y, x):
                if f[vy][vx] > g[y][x]:
                    g[y][x] = f[vy][vx]
    return g

def erode(f, B, H, W):
    g = [row[:] for row in f]
    for y in range(H):
        for x in range(W):
            for vy, vx in vizinhos(H, W, B, y, x):
                if f[vy][vx] < g[y][x]:
                    g[y][x] = f[vy][vx]
    return g

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

d = dilata(f, B, L, C)
e = erode(f, B, L, C)
abertura = dilata(e, B, L, C)
fechamento = erode(d, B, L, C)

gradiente = [[d[y][x] - e[y][x] for x in range(C)] for y in range(L)]
tophat = [[f[y][x] - abertura[y][x] for x in range(C)] for y in range(L)]
blackhat = [[fechamento[y][x] - f[y][x] for x in range(C)] for y in range(L)]

for row in gradiente:
    print(*row)
for row in tophat:
    print(*row)
for row in blackhat:
    print(*row)


Overwriting EP04_08.py


In [26]:
TestSuite("EP04_08.py").run()


---
### EP04_09 🗺️ Transformada de Distância e o "Miolo" do Objeto
Em **robótica móvel**, ao planejar uma rota dentro de um corredor, o robô quer saber não só *onde* há espaço livre, mas **quão longe** cada ponto livre está da parede mais próxima — o caminho mais seguro passa pelo "miolo" do corredor, e não pela beirada. A **transformada de distância** responde exatamente isso: cada pixel do objeto recebe o número de **camadas de erosão** que ele resiste antes de desaparecer, e o pixel de valor máximo é o ponto mais protegido — o coração geométrico do objeto.
Ver na @fig-04-sim-distancia uma simulação deste EP.
#### 📋 Diretrizes de Implementação
1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $H_B$ (linhas) e $W_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz binária $f$ (valores $0$ ou $1$), linha a linha.
5. **Inicialização:** Criar a matriz de distâncias $\text{dist}$, toda em $0$, e copiar $f$ para uma matriz auxiliar $\text{atual}$.
6. **Decascamento iterativo:** Repetir, enquanto $\text{atual}$ tiver algum pixel $1$:
   1. Incrementar o nível: $\text{nivel} \mathrel{+}= 1$.
   2. Para cada pixel onde $\text{atual}(y,x)=1$, atualizar $\text{dist}(y,x) = \text{nivel}$.
   3. Calcular $\text{prox} = \text{atual} \ominus B$ (erosão binária plana, exatamente como no EP04_04).
   4. Se $\text{prox}$ for **idêntico** a $\text{atual}$ (nenhum pixel mudou — situação de elemento estruturante incapaz de continuar encolhendo), **parar o laço**.
   5. Caso contrário, $\text{atual} \leftarrow \text{prox}$ e continuar.
7. **Localização do miolo:** Encontrar o valor máximo de $\text{dist}$ e a posição $(y^*,x^*)$ da **primeira** ocorrência desse máximo, percorrendo a matriz em ordem **raster** (linha a linha, da esquerda para a direita).
8. **Saída:** Exibir a matriz $\text{dist}$ completa e, na última linha, os dois inteiros $y^*$ e $x^*$ separados por espaço.
---
#### 📌 Restrições Computacionais
* **Erosão sem reflexão e sem padding**, idêntica à do EP04_04, em cada iteração.
* **Critério de parada duplo:** parar tanto quando não houver mais $1$s quanto quando a erosão deixar de mudar a matriz (evita laço infinito em casos degenerados).
* **Desempate determinístico:** entre múltiplos pixels com o valor máximo, escolher sempre o **primeiro em ordem raster**.
---
#### 🧠 Fundamentação Teórica
| Conceito | Significado | Impacto Visual |
|----------|-------------|-----------------|
| **$\text{dist}(y,x)$** | Número de erosões que o pixel sobrevive | Equivale à distância (em "passos" de $B$) até a borda mais próxima |
| **Valor máximo** | Pixel mais "protegido" do objeto | Aproxima o centro do **esqueleto morfológico** |
| **Objetos finos** | Largura comparável a $B$ | Toda a região tem o mesmo valor baixo de distância |
---
#### 📦 Especificação de Entrada e Saída (VPL)
**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $H_B$.
* Linha 4: Inteiro $W_B$.
* Próximas $H_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros ($0$ ou $1$) da matriz $f$.
**Saída:**
* Matriz $\text{dist}$ em $L$ linhas e $C$ colunas.
* Última linha: dois inteiros $y^*$ e $x^*$ (posição do miolo), separados por espaço.
#### 📌 Exemplos
| Entrada | Saída | Observação |
|---------|-------|------------|
| 5<br>9<br>3<br>3<br>0 1 0<br>1 1 1<br>0 1 0<br>0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0 | 0 0 0 0 0 0 0 0 0<br>0 1 1 1 1 1 1 1 0<br>0 1 2 2 2 2 2 1 0<br>0 1 1 1 1 1 1 1 0<br>0 0 0 0 0 0 0 0 0<br>2 2 | O "corredor" 3×7 tem miolo em toda a linha central; o primeiro em ordem raster é $(2,2)$ |


In [27]:
#| label: fig-04-sim-distancia
#| fig-cap: "Simulador: Transformada de Distância e o Miolo do Objeto"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0409" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Distância por Decascamento</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🗺️ camadas de erosão</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#555;margin-bottom:10px;text-align:center;">Clique nas células para desenhar seu próprio objeto (mínimo 3 pixels)</p>
    <div style="display:flex;justify-content:center;margin-bottom:18px;">
      <div id="ep0409_grid_f" style="display:grid;grid-template-columns:repeat(9,32px);gap:2px;"></div>
    </div>
    <div style="text-align:center;margin-bottom:14px;">
      <button id="ep0409_btn_corredor" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;margin-right:6px;">📐 Corredor</button>
      <button id="ep0409_btn_disco" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;margin-right:6px;">⬤ Disco</button>
      <button id="ep0409_btn_l" style="font-size:11px;padding:5px 10px;border-radius:4px;border:1px solid #ccc;background:white;cursor:pointer;">📏 Forma em L</button>
    </div>
    <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;text-align:center;">Mapa de distâncias (★ = miolo)</p>
    <div style="display:flex;justify-content:center;">
      <div id="ep0409_grid_dist" style="display:grid;grid-template-columns:repeat(9,32px);gap:2px;"></div>
    </div>
  </div>
</div>
<script>
setTimeout(function(){
  var gF=document.getElementById('ep0409_grid_f');
  var gD=document.getElementById('ep0409_grid_dist');
  if(!gF)return;
  var L=5,C=9,f=[];
  var B=[[0,1,0],[1,1,1],[0,1,0]];
  function setCorredor(){
    f=Array.from({length:L},function(){return new Array(C).fill(0);});
    for(var y=1;y<4;y++)for(var x=1;x<8;x++) f[y][x]=1;
  }
  function setDisco(){
    f=Array.from({length:L},function(){return new Array(C).fill(0);});
    var cy=2,cx=4;
    for(var y=0;y<L;y++)for(var x=0;x<C;x++) if(Math.pow(y-cy,2)+Math.pow((x-cx)*0.6,2)<=4) f[y][x]=1;
  }
  function setL(){
    f=Array.from({length:L},function(){return new Array(C).fill(0);});
    for(var y=1;y<4;y++)for(var x=1;x<3;x++) f[y][x]=1;
    for(var y=2;y<4;y++)for(var x=1;x<8;x++) f[y][x]=1;
  }
  function erode(img,Bm){
    var HB=Bm.length, WB=Bm[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
    var g=img.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++)for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
      if(Bm[by][bx]!==1) continue;
      var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
      if(vy>=0&&vy<L&&vx>=0&&vx<C && img[vy][vx]<g[y][x]) g[y][x]=img[vy][vx];
    }
    return g;
  }
  function sameMatrix(a,b){
    for(var y=0;y<L;y++)for(var x=0;x<C;x++) if(a[y][x]!==b[y][x]) return false;
    return true;
  }
  function render_0409(){
    gF.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      (function(y,x){
        var c=document.createElement('div');
        c.style.cssText='width:32px;height:32px;border-radius:4px;border:1px solid #ddd;cursor:pointer;';
        c.style.background=f[y][x]?'#16a085':'#f7f7f7';
        c.onclick=function(){f[y][x]=1-f[y][x]; render_0409();};
        gF.appendChild(c);
      })(y,x);
    }
    var dist=Array.from({length:L},function(){return new Array(C).fill(0);});
    var atual=f.map(function(r){return r.slice();});
    var nivel=0;
    while(atual.some(function(r){return r.some(function(v){return v===1;});})){
      nivel++;
      for(var y=0;y<L;y++)for(var x=0;x<C;x++) if(atual[y][x]===1) dist[y][x]=nivel;
      var prox=erode(atual,B);
      if(sameMatrix(prox,atual)) break;
      atual=prox;
    }
    var maxV=0;
    for(var y=0;y<L;y++)for(var x=0;x<C;x++) if(dist[y][x]>maxV) maxV=dist[y][x];
    var found=false, my=-1, mx=-1;
    for(var y=0;y<L && !found;y++)for(var x=0;x<C;x++) if(dist[y][x]===maxV && maxV>0){my=y;mx=x;found=true;break;}
    gD.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var c=document.createElement('div');
      var v=dist[y][x];
      var t=maxV>0?v/maxV:0;
      var inten=Math.round(220-t*170);
      c.style.cssText='width:32px;height:32px;border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:11px;font-weight:bold;border:1px solid #ddd;';
      c.style.background= v===0 ? '#f7f7f7' : 'rgb('+(inten-60)+','+inten+','+(inten-30)+')';
      c.style.color = v>0 ? 'white':'#bbb';
      c.innerText = (y===my && x===mx) ? '★' : (v||'');
      gD.appendChild(c);
    }
  }
  document.getElementById('ep0409_btn_corredor').onclick=function(){setCorredor();render_0409();};
  document.getElementById('ep0409_btn_disco').onclick=function(){setDisco();render_0409();};
  document.getElementById('ep0409_btn_l').onclick=function(){setL();render_0409();};
  setCorredor(); render_0409();
},100);
</script>
""")


In [28]:
%%writefile EP04_09.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

def erode(f, B, H, W):
    g = [row[:] for row in f]
    for y in range(H):
        for x in range(W):
            for vy, vx in vizinhos(H, W, B, y, x):
                if f[vy][vx] < g[y][x]:
                    g[y][x] = f[vy][vx]
    return g

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

dist = [[0] * C for _ in range(L)]
atual = [row[:] for row in f]
nivel = 0
while any(any(row) for row in atual):
    nivel += 1
    for y in range(L):
        for x in range(C):
            if atual[y][x] == 1:
                dist[y][x] = nivel
    prox = erode(atual, B, L, C)
    if prox == atual:
        break
    atual = prox

maxv = max(max(row) for row in dist)
pos_y, pos_x = 0, 0
encontrado = False
for y in range(L):
    for x in range(C):
        if dist[y][x] == maxv:
            pos_y, pos_x = y, x
            encontrado = True
            break
    if encontrado:
        break

for row in dist:
    print(*row)
print(pos_y, pos_x)


Overwriting EP04_09.py


In [29]:
TestSuite("EP04_09.py").run()


---
### EP04_10 🪙 Separação de Blobs, Rotulação e Descritores
Em uma **linha de produção de moedas**, é comum que peças encostem umas nas outras na esteira, formando uma única mancha conectada na imagem — uma contagem ingênua erraria o total. A solução clássica combina tudo o que foi construído neste capítulo: primeiro uma **erosão** rompe as "pontes" finas que ligam objetos vizinhos, depois a **rotulação por conectividade** (como em `mm.label0`) separa cada peça em um rótulo próprio, e por fim **descritores geométricos** (área e caixa delimitadora) resumem cada peça encontrada.
Ver na @fig-04-sim-rotulacao uma simulação deste EP.
#### 📋 Diretrizes de Implementação
1. **Dimensões da imagem:** Ler os inteiros $L$ (linhas) e $C$ (colunas) de $f$.
2. **Dimensões de $B$:** Ler os inteiros $H_B$ (linhas) e $W_B$ (colunas) do elemento estruturante.
3. **Elemento estruturante:** Ler a matriz $B$ com valores $0$ ou $1$, linha a linha.
4. **Dados:** Ler a matriz binária $f$ (valores $0$ ou $1$), linha a linha.
5. **Separação:** Calcular $f_{ero} = f \ominus B$ (erosão binária plana, igual ao EP04_04), eliminando pontes finas entre objetos.
6. **Rotulação ($K{=}8$):** Sobre $f_{ero}$, identificar componentes conectados usando **8-vizinhança** (as $8$ posições adjacentes, incluindo diagonais). Percorrer a imagem em ordem raster; ao encontrar um pixel $1$ ainda sem rótulo, atribuir um novo rótulo inteiro crescente a partir de $1$ e propagá-lo a toda a região conectada (busca em largura ou profundidade).
7. **Descritores:** Para cada rótulo $k$, calcular:
   * **Área:** número de pixels com aquele rótulo.
   * **Caixa delimitadora:** $(y_{min}, x_{min}, y_{max}, x_{max})$, as menores e maiores coordenadas de linha e coluna entre os pixels do rótulo.
8. **Saída:** Exibir o número total de rótulos encontrados e, em seguida, uma linha por rótulo (em ordem crescente) com: rótulo, área, $y_{min}$, $x_{min}$, $y_{max}$, $x_{max}$.
---
#### 📌 Restrições Computacionais
* **A erosão acontece antes da rotulação**, sempre — rotular $f$ diretamente (sem separar) pode fundir objetos em um só componente.
* **8-vizinhança fixa nesta etapa de rotulação**, independente do $B$ usado na erosão (que pode ter outro formato).
* **Sem padding em nenhuma etapa.**
* **Ordem dos rótulos:** seguir a ordem de **primeira descoberta** em varredura raster.
---
#### 🧠 Fundamentação Teórica
| Conceito | Significado | Impacto |
|----------|-------------|---------|
| **Ponte fina** | Conexão de espessura $1$ entre dois objetos | Some após uma erosão com $B$ maior que a espessura da ponte |
| **8-vizinhança** | Inclui diagonais | Reconecta cantos que a 4-vizinhança deixaria separados |
| **Área pequena** | Poucos pixels no rótulo | Pode indicar ruído residual, não um objeto real |
---
#### 📦 Especificação de Entrada e Saída (VPL)
**Entrada:**
* Linha 1: Inteiro $L$.
* Linha 2: Inteiro $C$.
* Linha 3: Inteiro $H_B$.
* Linha 4: Inteiro $W_B$.
* Próximas $H_B$ linhas: elementos inteiros ($0$ ou $1$) da matriz $B$.
* Próximas $L$ linhas: elementos inteiros ($0$ ou $1$) da matriz $f$.
**Saída:**
* Linha 1: número total de rótulos encontrados.
* Uma linha por rótulo, na ordem $k, \text{área}, y_{min}, x_{min}, y_{max}, x_{max}$.
#### 📌 Exemplos
| Entrada | Saída | Observação |
|---------|-------|------------|
| 7<br>10<br>3<br>3<br>1 1 1<br>1 1 1<br>1 1 1<br>0 0 0 0 0 0 0 0 0 0<br>0 1 1 1 0 0 1 1 1 0<br>0 1 1 1 0 0 1 1 1 0<br>0 1 1 1 1 1 1 1 1 0<br>0 1 1 1 0 0 1 1 1 0<br>0 1 1 1 0 0 1 1 1 0<br>0 0 0 0 0 0 0 0 0 0 | 2<br>1 3 2 2 4 2<br>2 3 2 7 4 7 | A ponte central de $1$ pixel de espessura é removida pela erosão $3\times3$; os dois blocos $3\times3$ encolhem para colunas únicas e são contados separadamente |


In [30]:
#| label: fig-04-sim-rotulacao
#| fig-cap: "Simulador: Separação de Blobs, Rotulação e Descritores"
#| echo: false
#| output: true
from IPython.display import HTML
HTML("""
<div id="sim-ep0410" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Moedas Coladas → Separadas → Contadas</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🪙 erosão + rótulo + descritores</span>
  </div>
  <div style="padding:20px;background:white;">
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:15px;margin-bottom:18px;text-align:center;">
      <label style="font-size:12px;font-weight:bold;color:#b9770e;">Espessura da ponte entre as moedas</label><br>
      <input id="ep0410_sl_p" style="width:60%;accent-color:#b9770e;margin-top:8px;" max="3" min="1" step="1" type="range" value="1">
      <span id="ep0410_vl_p" style="font-family:monospace;font-weight:bold;color:#b9770e;margin-left:8px;">1 px</span>
    </div>
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:14px;">
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">f original (ligadas)</p>
        <div id="ep0410_grid_f" style="display:grid;grid-template-columns:repeat(10,26px);gap:2px;justify-content:center;"></div>
      </div>
      <div style="text-align:center;background:white;border:1px solid #eee;padding:12px;border-radius:12px;">
        <p style="font-size:10px;font-weight:600;color:#888;text-transform:uppercase;margin-bottom:10px;">após erosão + rótulos</p>
        <div id="ep0410_grid_lab" style="display:grid;grid-template-columns:repeat(10,26px);gap:2px;justify-content:center;"></div>
      </div>
    </div>
    <div id="ep0410_info" style="margin-top:18px;background:#fff8e1;border-radius:8px;padding:12px;border:1px solid #ffe0a3;font-size:12px;color:#7d5a00;text-align:center;"></div>
  </div>
</div>
<script>
setTimeout(function(){
  var slP=document.getElementById('ep0410_sl_p');
  var vlP=document.getElementById('ep0410_vl_p');
  var gF=document.getElementById('ep0410_grid_f');
  var gL=document.getElementById('ep0410_grid_lab');
  var info=document.getElementById('ep0410_info');
  if(!slP||!gF)return;
  var L=7,C=10;
  var B=[[1,1,1],[1,1,1],[1,1,1]];
  var palette=['#e74c3c','#27ae60','#2980b9','#8e44ad','#d35400'];
  function buildF(p){
    var f=Array.from({length:L},function(){return new Array(C).fill(0);});
    for(var y=1;y<6;y++)for(var x=1;x<4;x++) f[y][x]=1;
    for(var y=1;y<6;y++)for(var x=6;x<9;x++) f[y][x]=1;
    var midRow=3;
    for(var dy=0;dy<p;dy++){
      var ry=midRow-Math.floor(p/2)+dy;
      for(var x=4;x<6;x++) f[ry][x]=1;
    }
    return f;
  }
  function erode(img,Bm){
    var HB=Bm.length, WB=Bm[0].length, oy=-HB/2+0.5, ox=-WB/2+0.5;
    var g=img.map(function(r){return r.slice();});
    for(var y=0;y<L;y++)for(var x=0;x<C;x++)for(var by=0;by<HB;by++)for(var bx=0;bx<WB;bx++){
      if(Bm[by][bx]!==1) continue;
      var vy=Math.trunc(y+by+oy), vx=Math.trunc(x+bx+ox);
      if(vy>=0&&vy<L&&vx>=0&&vx<C && img[vy][vx]<g[y][x]) g[y][x]=img[vy][vx];
    }
    return g;
  }
  function labelK8(img){
    var labels=Array.from({length:L},function(){return new Array(C).fill(0);});
    var dirs=[[-1,-1],[-1,0],[-1,1],[0,-1],[0,1],[1,-1],[1,0],[1,1]];
    var cur=0, desc=[];
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      if(img[y][x]===1 && labels[y][x]===0){
        cur++;
        var stack=[[y,x]];
        labels[y][x]=cur;
        var area=0, miny=y,maxy=y,minx=x,maxx=x;
        while(stack.length){
          var cell=stack.pop(); var cy=cell[0],cx=cell[1];
          area++;
          if(cy<miny)miny=cy; if(cy>maxy)maxy=cy;
          if(cx<minx)minx=cx; if(cx>maxx)maxx=cx;
          dirs.forEach(function(d){
            var ny=cy+d[0],nx=cx+d[1];
            if(ny>=0&&ny<L&&nx>=0&&nx<C&&img[ny][nx]===1&&labels[ny][nx]===0){
              labels[ny][nx]=cur; stack.push([ny,nx]);
            }
          });
        }
        desc.push({k:cur,area:area,miny:miny,minx:minx,maxy:maxy,maxx:maxx});
      }
    }
    return {labels:labels, desc:desc};
  }
  function paintBin(grid,img){
    grid.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var c=document.createElement('div');
      c.style.cssText='width:26px;height:26px;border-radius:3px;border:1px solid #ddd;';
      c.style.background=img[y][x]?'#b9770e':'#fdf6ec';
      grid.appendChild(c);
    }
  }
  function paintLabels(grid,labels){
    grid.innerHTML='';
    for(var y=0;y<L;y++)for(var x=0;x<C;x++){
      var c=document.createElement('div');
      var k=labels[y][x];
      c.style.cssText='width:26px;height:26px;border-radius:3px;border:1px solid #ddd;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:bold;color:white;';
      c.style.background = k>0 ? palette[(k-1)%palette.length] : '#fdf6ec';
      c.innerText = k>0 ? k : '';
      grid.appendChild(c);
    }
  }
  function render_0410(){
    var p=parseInt(slP.value);
    vlP.innerText=p+' px';
    var f=buildF(p);
    var fe=erode(f,B);
    var res=labelK8(fe);
    paintBin(gF,f);
    paintLabels(gL,res.labels);
    var txt='<b>'+res.desc.length+' objeto(s) detectado(s) após a erosão.</b><br>';
    res.desc.forEach(function(d){
      txt+='Rótulo '+d.k+': área='+d.area+', bbox=('+d.miny+','+d.minx+')→('+d.maxy+','+d.maxx+')<br>';
    });
    if(res.desc.length<2){
      txt+='<i>A ponte ainda é espessa demais para a erosão 3×3 — as moedas continuam fundidas em 1 só objeto.</i>';
    }
    info.innerHTML=txt;
  }
  slP.oninput=render_0410;
  render_0410();
},100);
</script>
""")


In [31]:
%%writefile EP04_10.py
def vizinhos(H, W, B, y, x):
    HB, WB = len(B), len(B[0])
    oy = -HB / 2 + 0.5
    ox = -WB / 2 + 0.5
    for by in range(HB):
        for bx in range(WB):
            vy = int(y + by + oy)
            vx = int(x + bx + ox)
            if 0 <= vy < H and 0 <= vx < W and B[by][bx] == 1:
                yield vy, vx

def erode(f, B, H, W):
    g = [row[:] for row in f]
    for y in range(H):
        for x in range(W):
            for vy, vx in vizinhos(H, W, B, y, x):
                if f[vy][vx] < g[y][x]:
                    g[y][x] = f[vy][vx]
    return g

L = int(input())
C = int(input())
HB = int(input())
WB = int(input())
B = [list(map(int, input().split())) for _ in range(HB)]
f = [list(map(int, input().split())) for _ in range(L)]

f_erodida = erode(f, B, L, C)

dirs8 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
labels = [[0] * C for _ in range(L)]
desc = []
atual = 0
for y in range(L):
    for x in range(C):
        if f_erodida[y][x] == 1 and labels[y][x] == 0:
            atual += 1
            pilha = [(y, x)]
            labels[y][x] = atual
            area = 0
            min_y = max_y = y
            min_x = max_x = x
            while pilha:
                cy, cx = pilha.pop()
                area += 1
                if cy < min_y:
                    min_y = cy
                if cy > max_y:
                    max_y = cy
                if cx < min_x:
                    min_x = cx
                if cx > max_x:
                    max_x = cx
                for dy, dx in dirs8:
                    ny, nx = cy + dy, cx + dx
                    if 0 <= ny < L and 0 <= nx < C and f_erodida[ny][nx] == 1 and labels[ny][nx] == 0:
                        labels[ny][nx] = atual
                        pilha.append((ny, nx))
            desc.append((atual, area, min_y, min_x, max_y, max_x))

print(len(desc))
for rotulo, area, min_y, min_x, max_y, max_x in desc:
    print(rotulo, area, min_y, min_x, max_y, max_x)


Overwriting EP04_10.py


In [32]:
TestSuite("EP04_10.py").run()
